# Emotion-Aware Tic-Tac-Toe

This notebook mirrors the current `main.py` / `emotion_game_ai.runtime.run_app` entrypoint so you can launch and experiment with the game from a single place.

Run the cells top-to-bottom to ensure imports and paths are set up correctly before starting the app.


In [ ]:
# Environment and dependencies (Colab-friendly)

import os

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Optionally change this path if you want to read assets/models from Drive.
    PROJECT_ROOT = "/content/drive/MyDrive/TicTacToe"
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.chdir(PROJECT_ROOT)

# Install third-party dependencies. This does NOT depend on local .py files.
%pip install -q numpy pandas scikit-learn nltk mediapipe opencv-python pygame


In [ ]:
# Emotion behavior mapping (inline from emotion_behavior.py)

from dataclasses import dataclass

@dataclass(frozen=True)
class EmotionBehavior:
    emotion: str
    ai_difficulty: str
    ai_personality: str
    ui_theme: str
    animation_intensity: float
    hint_frequency: float
    ai_thinking_time: float
    system_message: str
    ui_effect: str


_BEHAVIORS: dict[str, EmotionBehavior] = {
    "neutral": EmotionBehavior(
        emotion="neutral",
        ai_difficulty="balanced",
        ai_personality="standard",
        ui_theme="calm_blue",
        animation_intensity=0.35,
        hint_frequency=0.35,
        ai_thinking_time=1.1,
        system_message="Let's continue playing.",
        ui_effect="none",
    ),
    "love": EmotionBehavior(
        emotion="love",
        ai_difficulty="slightly_increased",
        ai_personality="friendly_competitive",
        ui_theme="warm_pink",
        animation_intensity=0.80,
        hint_frequency=0.15,
        ai_thinking_time=0.7,
        system_message="I'm glad you're enjoying the game!",
        ui_effect="hearts",
    ),
    "happiness": EmotionBehavior(
        emotion="happiness",
        ai_difficulty="competitive",
        ai_personality="playful",
        ui_theme="bright_glow",
        animation_intensity=0.95,
        hint_frequency=0.15,
        ai_thinking_time=0.65,
        system_message="You seem happy! Let's make the next round exciting.",
        ui_effect="board_glow",
    ),
    "sadness": EmotionBehavior(
        emotion="sadness",
        ai_difficulty="reduced",
        ai_personality="supportive",
        ui_theme="soft_blue",
        animation_intensity=0.20,
        hint_frequency=0.75,
        ai_thinking_time=1.4,
        system_message="Don't worry. Every move is a chance to improve.",
        ui_effect="calm_bg",
    ),
    "relief": EmotionBehavior(
        emotion="relief",
        ai_difficulty="balanced",
        ai_personality="calm",
        ui_theme="calm_blue",
        animation_intensity=0.25,
        hint_frequency=0.35,
        ai_thinking_time=1.0,
        system_message="Looks like you're feeling better.",
        ui_effect="slow_fade",
    ),
    "hate": EmotionBehavior(
        emotion="hate",
        ai_difficulty="significantly_reduced",
        ai_personality="apologetic_helpful",
        ui_theme="supportive",
        animation_intensity=0.15,
        hint_frequency=0.95,
        ai_thinking_time=1.6,
        system_message="I'm sorry the game felt frustrating. Let's try an easier round.",
        ui_effect="supportive_theme",
    ),
    "anger": EmotionBehavior(
        emotion="anger",
        ai_difficulty="reduced",
        ai_personality="deescalating",
        ui_theme="calm_blue",
        animation_intensity=0.20,
        hint_frequency=0.70,
        ai_thinking_time=1.5,
        system_message="Take a moment. Let's play at a calmer pace.",
        ui_effect="calming_overlay",
    ),
    "fun": EmotionBehavior(
        emotion="fun",
        ai_difficulty="slightly_increased",
        ai_personality="playful",
        ui_theme="party",
        animation_intensity=1.00,
        hint_frequency=0.20,
        ai_thinking_time=0.75,
        system_message="Looks like you're having fun!",
        ui_effect="particles",
    ),
    "enthusiasm": EmotionBehavior(
        emotion="enthusiasm",
        ai_difficulty="maximum",
        ai_personality="highly_competitive",
        ui_theme="dynamic",
        animation_intensity=1.00,
        hint_frequency=0.10,
        ai_thinking_time=0.55,
        system_message="You seem excited! Let's push your skills.",
        ui_effect="dynamic_lighting",
    ),
    "surprise": EmotionBehavior(
        emotion="surprise",
        ai_difficulty="balanced",
        ai_personality="curious",
        ui_theme="flash",
        animation_intensity=0.70,
        hint_frequency=0.30,
        ai_thinking_time=1.0,
        system_message="That was unexpected! Let's see what happens next.",
        ui_effect="flash",
    ),
    "empty": EmotionBehavior(
        emotion="empty",
        ai_difficulty="balanced",
        ai_personality="encouraging",
        ui_theme="subtle_pulse",
        animation_intensity=0.30,
        hint_frequency=0.45,
        ai_thinking_time=1.1,
        system_message="Let's keep the game engaging.",
        ui_effect="subtle_pulse",
    ),
    "worry": EmotionBehavior(
        emotion="worry",
        ai_difficulty="reduced",
        ai_personality="reassuring",
        ui_theme="soft_blue",
        animation_intensity=0.20,
        hint_frequency=0.80,
        ai_thinking_time=1.4,
        system_message="It's okay to take your time.",
        ui_effect="calm_bg",
    ),
    "boredom": EmotionBehavior(
        emotion="boredom",
        ai_difficulty="unpredictable",
        ai_personality="entertaining",
        ui_theme="dynamic",
        animation_intensity=1.00,
        hint_frequency=0.10,
        ai_thinking_time=0.8,
        system_message="Let's make things more interesting!",
        ui_effect="dynamic_board",
    ),
}


def behavior_for_emotion(emotion: str) -> EmotionBehavior:
    key = (emotion or "").strip().lower() or "neutral"
    return _BEHAVIORS.get(key, _BEHAVIORS["neutral"])


In [ ]:
# Threading utilities and shared state (inline from utils/threading_utils.py)

from collections import Counter, deque
import threading
import time
from dataclasses import dataclass, field
from typing import Deque, Optional

EmotionLabel = str  # "Happy" | "Neutral"


class StoppableThread(threading.Thread):
    """Thread with a cooperative stop event."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.stop_event = threading.Event()

    def stop(self) -> None:
        self.stop_event.set()


@dataclass
class RollingMajority:
    """Rolling majority vote for categorical labels."""

    window_size: int = 30
    _items: Deque[EmotionLabel] = field(default_factory=deque, init=False)

    def add(self, label: EmotionLabel) -> None:
        self._items.append(label)
        while len(self._items) > self.window_size:
            self._items.popleft()

    def majority(self, default: EmotionLabel = "Neutral") -> EmotionLabel:
        if not self._items:
            return default
        counts = Counter(self._items)
        return counts.most_common(1)[0][0]


@dataclass
class VisionSnapshot:
    """Latest vision outputs for UI consumption."""

    emotion_raw: EmotionLabel = "Neutral"
    emotion_smoothed: EmotionLabel = "Neutral"
    mouth_width: float = 0.0
    landmarks_detected: int = 0
    fps: float = 0.0
    camera_index: int = 0
    camera_ok: bool = False
    # RGB frame for UI preview (numpy uint8 HxWx3), optional
    preview_rgb: Optional[object] = None


@dataclass
class GameStats:
    games_played: int = 0
    wins: int = 0
    losses: int = 0
    draws: int = 0
    emotion_counts: dict[str, int] = field(default_factory=lambda: {"Happy": 0, "Neutral": 0})

    def record_emotion(self, emotion: EmotionLabel) -> None:
        self.emotion_counts[emotion] = self.emotion_counts.get(emotion, 0) + 1


@dataclass
class DifficultyTuning:
    """Cross-match tuning updated via sentiment."""

    assistive_mistake_prob: float = 0.22  # base for Neutral mode
    ai_thinking_delay_s: float = 1.2
    hint_frequency: float = 0.35
    ai_personality: str = "standard"
    ui_theme: str = "calm_blue"

    def adjust_for_sentiment(self, sentiment: str) -> None:
        if sentiment == "negative":
            self.assistive_mistake_prob = min(0.35, self.assistive_mistake_prob + 0.03)
        elif sentiment == "positive":
            self.assistive_mistake_prob = max(0.10, self.assistive_mistake_prob - 0.03)

    def apply_behavior(self, emotion: str) -> EmotionBehavior:
        """Apply a full gameplay response profile derived from an emotion label."""
        b = behavior_for_emotion(emotion)
        self.ai_thinking_delay_s = float(b.ai_thinking_time)
        self.hint_frequency = float(b.hint_frequency)
        self.ai_personality = b.ai_personality
        self.ui_theme = b.ui_theme

        if b.ai_difficulty in {"significantly_reduced"}:
            self.assistive_mistake_prob = 0.33
        elif b.ai_difficulty in {"reduced"}:
            self.assistive_mistake_prob = 0.28
        elif b.ai_difficulty in {"balanced"}:
            self.assistive_mistake_prob = 0.22
        elif b.ai_difficulty in {"slightly_increased"}:
            self.assistive_mistake_prob = 0.18
        elif b.ai_difficulty in {"competitive", "maximum"}:
            self.assistive_mistake_prob = 0.12
        elif b.ai_difficulty in {"unpredictable"}:
            self.assistive_mistake_prob = 0.16
        return b


class SharedState:
    """Thread-safe state shared between vision thread and game/UI."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.vision = VisionSnapshot()
        self.stats = GameStats()
        self.tuning = DifficultyTuning()
        self.current_emotion: EmotionLabel = "Neutral"
        self.last_feedback_emotion: str = "neutral"
        self.last_behavior: EmotionBehavior = behavior_for_emotion("neutral")
        self.ai_mode: str = "Assistive"
        self.dialogue_message: str = ""
        self.dialogue_until_s: float = 0.0
        self.available_cameras: list[int] = [0]
        self._requested_camera_index: int = 0
        self._max_camera_index: int = 4

    def set_dialogue(self, message: str, ttl_s: float = 3.0) -> None:
        now = time.perf_counter()
        with self._lock:
            self.dialogue_message = message
            self.dialogue_until_s = now + ttl_s

    def get_dialogue(self) -> str:
        now = time.perf_counter()
        with self._lock:
            if self.dialogue_message and now <= self.dialogue_until_s:
                return self.dialogue_message
            return ""

    def update_vision(self, snap: VisionSnapshot) -> None:
        with self._lock:
            self.vision = snap
            self.current_emotion = snap.emotion_smoothed

    def get_snapshot(self) -> tuple[EmotionLabel, VisionSnapshot, GameStats, DifficultyTuning, str]:
        with self._lock:
            return (
                self.current_emotion,
                self.vision,
                self.stats,
                self.tuning,
                self.ai_mode,
            )

    def set_available_cameras(self, indices: list[int]) -> None:
        cleaned = sorted({int(i) for i in indices if i is not None and int(i) >= 0})
        if not cleaned:
            cleaned = [0]
        with self._lock:
            self.available_cameras = cleaned
            if self._requested_camera_index not in self.available_cameras:
                self._requested_camera_index = self.available_cameras[0]

    def request_camera(self, camera_index: int) -> None:
        with self._lock:
            self._requested_camera_index = int(camera_index)

    def request_next_camera(self) -> int:
        with self._lock:
            self._requested_camera_index = (
                (self._requested_camera_index + 1) % (self._max_camera_index + 1)
            )
            return self._requested_camera_index

    def get_requested_camera(self) -> int:
        with self._lock:
            return int(self._requested_camera_index)

    def camera_status_summary(self, max_index: int = 4) -> str:
        with self._lock:
            ok_set = set(int(i) for i in self.available_cameras)
        parts: list[str] = []
        for idx in range(0, int(max_index) + 1):
            label = "OK" if idx in ok_set else "N/A"
            parts.append(f"{idx}({label})")
        return ", ".join(parts)

    def set_feedback_emotion(self, emotion: str) -> EmotionBehavior:
        with self._lock:
            self.last_feedback_emotion = (emotion or "").strip().lower() or "neutral"
            self.last_behavior = self.tuning.apply_behavior(self.last_feedback_emotion)
            return self.last_behavior


In [ ]:
# Tic-Tac-Toe board and minimax AI (inline from board.py and minimax.py)

from dataclasses import dataclass, field
from typing import Optional

PlayerMark = str  # "X" | "O"


@dataclass
class Board:
    grid: list[list[str]] = field(default_factory=lambda: [["", "", ""], ["", "", ""], ["", "", ""]])

    def copy(self) -> "Board":
        b = Board()
        b.grid = [row[:] for row in self.grid]
        return b

    def empty_cells(self) -> list[tuple[int, int]]:
        cells: list[tuple[int, int]] = []
        for r in range(3):
            for c in range(3):
                if self.grid[r][c] == "":
                    cells.append((r, c))
        return cells

    def place(self, r: int, c: int, mark: PlayerMark) -> bool:
        if self.grid[r][c] != "":
            return False
        self.grid[r][c] = mark
        return True

    def winner(self) -> Optional[PlayerMark]:
        lines: list[list[tuple[int, int]]] = []
        lines.extend([[(r, 0), (r, 1), (r, 2)] for r in range(3)])
        lines.extend([[(0, c), (1, c), (2, c)] for c in range(3)])
        lines.append([(0, 0), (1, 1), (2, 2)])
        lines.append([(0, 2), (1, 1), (2, 0)])

        for line in lines:
            vals = [self.grid[r][c] for r, c in line]
            if vals[0] and vals[0] == vals[1] == vals[2]:
                return vals[0]
        return None

    def is_draw(self) -> bool:
        return self.winner() is None and all(self.grid[r][c] for r in range(3) for c in range(3))

    def game_over(self) -> bool:
        return self.winner() is not None or self.is_draw()

    @staticmethod
    def winning_line_coords(grid: list[list[str]]) -> Optional[list[tuple[int, int]]]:
        lines: list[list[tuple[int, int]]] = []
        lines.extend([[(r, 0), (r, 1), (r, 2)] for r in range(3)])
        lines.extend([[(0, c), (1, c), (2, c)] for c in range(3)])
        lines.append([(0, 0), (1, 1), (2, 2)])
        lines.append([(0, 2), (1, 1), (2, 0)])
        for line in lines:
            vals = [grid[r][c] for r, c in line]
            if vals[0] and vals[0] == vals[1] == vals[2]:
                return line
        return None


AI_MARK = "O"
HUMAN_MARK = "X"


def _evaluate(board: Board) -> int:
    winner = board.winner()
    if winner == AI_MARK:
        return 10
    if winner == HUMAN_MARK:
        return -10
    return 0


def minimax(board: Board, depth: int, is_maximizing: bool) -> int:
    score = _evaluate(board)
    if score != 0:
        return score - depth if score > 0 else score + depth
    if board.is_draw():
        return 0

    if is_maximizing:
        best = -10_000
        for r, c in board.empty_cells():
            b2 = board.copy()
            b2.place(r, c, AI_MARK)
            best = max(best, minimax(b2, depth + 1, False))
        return best

    best = 10_000
    for r, c in board.empty_cells():
        b2 = board.copy()
        b2.place(r, c, HUMAN_MARK)
        best = min(best, minimax(b2, depth + 1, True))
    return best


def get_best_move(board: Board) -> tuple[int, int]:
    best_val = -10_000
    best_move = (-1, -1)
    for r, c in board.empty_cells():
        b2 = board.copy()
        b2.place(r, c, AI_MARK)
        move_val = minimax(b2, 0, False)
        if move_val > best_val:
            best_val = move_val
            best_move = (r, c)
    return best_move


In [ ]:
# Emotion-aware AI policy (inline from ai_player.py)

import random
from dataclasses import dataclass


@dataclass
class AIDecision:
    move: tuple[int, int]
    mode: str  # "Competitive" | "Assistive"
    thinking_delay_s: float
    message: str


def _suboptimal_move(board: Board) -> tuple[int, int]:
    """Choose a legal move that is not the best move when possible."""
    best = get_best_move(board)
    empties = board.empty_cells()
    if len(empties) <= 1:
        return best
    candidates = [m for m in empties if m != best]
    if not candidates:
        return best

    scored: list[tuple[int, tuple[int, int]]] = []
    for r, c in candidates:
        b2 = board.copy()
        b2.place(r, c, AI_MARK)
        val = minimax(b2, 0, False)
        scored.append((val, (r, c)))
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1]


def decide_move(
    board: Board,
    emotion: EmotionLabel,
    tuning: DifficultyTuning,
    rng: random.Random,
) -> AIDecision:
    if emotion == "Happy" and tuning.assistive_mistake_prob <= 0.14:
        return AIDecision(
            move=get_best_move(board),
            mode="Competitive",
            thinking_delay_s=max(0.35, min(0.9, tuning.ai_thinking_delay_s)),
            message="You look confident. Let’s see if you can beat me.",
        )

    mistake_prob = max(0.05, min(0.45, tuning.assistive_mistake_prob))
    if rng.random() < mistake_prob:
        move = _suboptimal_move(board)
    else:
        move = get_best_move(board)

    personality = getattr(tuning, "ai_personality", "standard")
    if personality in {"supportive", "reassuring", "deescalating"}:
        msg = "Take your time. I'm here to help you improve."
    elif personality in {"friendly_competitive", "highly_competitive"}:
        msg = "Let’s make this round a challenge."
    elif personality in {"entertaining", "playful"}:
        msg = "Alright! Let’s keep it interesting."
    elif personality in {"apologetic_helpful"}:
        msg = "Sorry that felt rough. Let’s slow down and find a good move."
    else:
        msg = "Let's keep going."

    return AIDecision(
        move=move,
        mode="Assistive",
        thinking_delay_s=max(0.35, min(2.0, tuning.ai_thinking_delay_s)),
        message=msg,
    )


In [ ]:
# Rendering helpers and animation engine (inline from renderer.py)

import math
from dataclasses import dataclass, field
from typing import Callable, Optional

import pygame


Color = pygame.Color


def clamp01(x: float) -> float:
    return 0.0 if x < 0.0 else 1.0 if x > 1.0 else x


def lerp(a: float, b: float, t: float) -> float:
    return a + (b - a) * t


def lerp_color(a: Color, b: Color, t: float) -> Color:
    t = clamp01(t)
    return Color(
        int(lerp(a.r, b.r, t)),
        int(lerp(a.g, b.g, t)),
        int(lerp(a.b, b.b, t)),
        int(lerp(a.a, b.a, t)),
    )


def ease_in_out(t: float) -> float:
    t = clamp01(t)
    return t * t * (3 - 2 * t)


def ease_in_cubic(t: float) -> float:
    t = clamp01(t)
    return t * t * t


def ease_out_back(t: float) -> float:
    t = clamp01(t)
    c1 = 1.70158
    c3 = c1 + 1
    return 1 + c3 * (t - 1) ** 3 + c1 * (t - 1) ** 2


@dataclass
class Tween:
    duration_s: float
    easing: Callable[[float], float] = ease_in_out
    elapsed_s: float = 0.0
    done: bool = False

    def reset(self) -> None:
        self.elapsed_s = 0.0
        self.done = False

    def step(self, dt_s: float) -> float:
        if self.done:
            return 1.0
        self.elapsed_s += max(0.0, dt_s)
        t = 1.0 if self.duration_s <= 0 else self.elapsed_s / self.duration_s
        if t >= 1.0:
            self.done = True
            t = 1.0
        return self.easing(t)


@dataclass
class Particle:
    pos: pygame.Vector2
    vel: pygame.Vector2
    radius: float
    color: Color
    life_s: float
    age_s: float = 0.0

    def step(self, dt_s: float) -> bool:
        self.age_s += dt_s
        self.pos += self.vel * dt_s
        self.vel.y += 420.0 * dt_s
        return self.age_s < self.life_s

    def draw(self, surf: pygame.Surface) -> None:
        t = clamp01(self.age_s / max(1e-6, self.life_s))
        alpha = int(255 * (1.0 - t))
        c = Color(self.color.r, self.color.g, self.color.b, alpha)
        pygame.draw.circle(surf, c, (int(self.pos.x), int(self.pos.y)), int(max(1, self.radius)))


@dataclass
class ParticleSystem:
    particles: list[Particle] = field(default_factory=list)

    def burst(self, center: tuple[int, int], palette: list[Color], count: int = 30) -> None:
        cx, cy = center
        for _ in range(count):
            angle = random.random() * math.tau
            speed = random.uniform(80, 260)
            vel = pygame.Vector2(math.cos(angle), math.sin(angle)) * speed
            p = Particle(
                pos=pygame.Vector2(cx, cy),
                vel=vel,
                radius=random.uniform(2.0, 4.5),
                color=random.choice(palette),
                life_s=random.uniform(0.6, 1.2),
            )
            self.particles.append(p)

    def step(self, dt_s: float) -> None:
        alive: list[Particle] = []
        for p in self.particles:
            if p.step(dt_s):
                alive.append(p)
        self.particles = alive

    def draw(self, surf: pygame.Surface) -> None:
        for p in self.particles:
            p.draw(surf)


@dataclass
class Theme:
    name: str
    accent: Color
    accent_soft: Color
    bg_top: Color
    bg_bottom: Color
    board_glow: Color


HAPPY_THEME = Theme(
    name="Happy",
    accent=Color(60, 220, 120),
    accent_soft=Color(120, 255, 180),
    bg_top=Color(14, 28, 20),
    bg_bottom=Color(10, 18, 14),
    board_glow=Color(70, 255, 150),
)

NEUTRAL_THEME = Theme(
    name="Neutral",
    accent=Color(80, 160, 255),
    accent_soft=Color(150, 210, 255),
    bg_top=Color(10, 16, 28),
    bg_bottom=Color(8, 12, 20),
    board_glow=Color(120, 190, 255),
)


EMOTION_THEMES: dict[str, Theme] = {
    "calm_blue": NEUTRAL_THEME,
    "soft_blue": Theme(
        name="Soft Blue",
        accent=Color(110, 180, 255),
        accent_soft=Color(180, 225, 255),
        bg_top=Color(10, 18, 34),
        bg_bottom=Color(8, 12, 22),
        board_glow=Color(160, 220, 255),
    ),
    "warm_pink": Theme(
        name="Warm Pink",
        accent=Color(255, 110, 180),
        accent_soft=Color(255, 170, 215),
        bg_top=Color(30, 14, 24),
        bg_bottom=Color(18, 8, 14),
        board_glow=Color(255, 150, 210),
    ),
    "bright_glow": Theme(
        name="Bright Glow",
        accent=Color(255, 215, 90),
        accent_soft=Color(255, 235, 150),
        bg_top=Color(22, 20, 10),
        bg_bottom=Color(14, 12, 8),
        board_glow=Color(255, 235, 140),
    ),
    "party": Theme(
        name="Party",
        accent=Color(170, 120, 255),
        accent_soft=Color(210, 180, 255),
        bg_top=Color(14, 10, 22),
        bg_bottom=Color(8, 6, 14),
        board_glow=Color(210, 170, 255),
    ),
    "dynamic": Theme(
        name="Dynamic",
        accent=Color(80, 240, 220),
        accent_soft=Color(150, 255, 240),
        bg_top=Color(8, 18, 18),
        bg_bottom=Color(6, 12, 12),
        board_glow=Color(130, 255, 240),
    ),
    "supportive": Theme(
        name="Supportive",
        accent=Color(120, 210, 190),
        accent_soft=Color(170, 240, 225),
        bg_top=Color(8, 18, 22),
        bg_bottom=Color(6, 12, 16),
        board_glow=Color(150, 245, 230),
    ),
    "flash": Theme(
        name="Flash",
        accent=Color(255, 255, 255),
        accent_soft=Color(220, 235, 255),
        bg_top=Color(10, 16, 28),
        bg_bottom=Color(8, 12, 20),
        board_glow=Color(200, 220, 255),
    ),
    "subtle_pulse": Theme(
        name="Subtle Pulse",
        accent=Color(150, 200, 255),
        accent_soft=Color(190, 230, 255),
        bg_top=Color(10, 16, 28),
        bg_bottom=Color(8, 12, 20),
        board_glow=Color(140, 210, 255),
    ),
}


def draw_vertical_gradient(surf: pygame.Surface, rect: pygame.Rect, top: Color, bottom: Color) -> None:
    h = max(1, rect.height)
    for i in range(h):
        t = i / float(h - 1) if h > 1 else 1.0
        c = lerp_color(top, bottom, t)
        pygame.draw.line(surf, c, (rect.left, rect.top + i), (rect.right - 1, rect.top + i))


def try_load_sound(path: str) -> Optional[pygame.mixer.Sound]:
    try:
        if not pygame.mixer.get_init():
            pygame.mixer.init()
        return pygame.mixer.Sound(path)
    except Exception:
        return None


In [ ]:
# Vision emotion classifier helpers (inline from vision/emotion_classifier.py)

from dataclasses import dataclass as _dataclass_ec
from typing import Tuple as _Tuple_ec

import numpy as _np_ec

LEFT_MOUTH_IDX = 61
RIGHT_MOUTH_IDX = 291


@_dataclass_ec(frozen=True)
class Calibration:
    baseline_mouth_width: float
    threshold: float


def mouth_width_px(landmarks_xy: _np_ec.ndarray, image_shape_hw: _Tuple_ec[int, int]) -> float:
    h, w = image_shape_hw
    p1 = landmarks_xy[LEFT_MOUTH_IDX]
    p2 = landmarks_xy[RIGHT_MOUTH_IDX]
    dx = (p1[0] - p2[0]) * w
    dy = (p1[1] - p2[1]) * h
    return float((dx * dx + dy * dy) ** 0.5)


def classify_emotion(mouth_width: float, threshold: float) -> str:
    return "Happy" if mouth_width > threshold else "Neutral"


In [ ]:
# NLP preprocessing and sentiment model (inline from nlp/preprocess.py and nlp/sentiment_model.py)

import re
from typing import Iterable

import nltk
import os as _os_sm
import pickle as _pickle_sm
from dataclasses import dataclass as _dataclass_sm
from typing import Optional as _Optional_sm

import pandas as _pd_sm
from sklearn.feature_extraction.text import TfidfVectorizer as _TfidfVectorizer
from sklearn.linear_model import LogisticRegression as _LogisticRegression
from sklearn.metrics import accuracy_score as _accuracy_score, classification_report as _classification_report, confusion_matrix as _confusion_matrix
from sklearn.model_selection import train_test_split as _train_test_split

_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_NON_WORD_RE = re.compile(r"[^a-z0-9\s]+", re.IGNORECASE)
_MULTI_SPACE_RE = re.compile(r"\s+")


def ensure_nltk() -> None:
    for pkg in ("punkt", "stopwords"):
        try:
            nltk.data.find(f"tokenizers/{pkg}" if pkg == "punkt" else f"corpora/{pkg}")
        except LookupError:
            nltk.download(pkg, quiet=True)


def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = _URL_RE.sub(" ", text)
    text = _NON_WORD_RE.sub(" ", text)
    text = _MULTI_SPACE_RE.sub(" ", text).strip()
    return text


def tokenize_and_remove_stopwords(text: str, stopwords: set[str]) -> str:
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stopwords and len(t) > 1]
    return " ".join(tokens)


def preprocess_texts(texts: Iterable[str]) -> list[str]:
    ensure_nltk()
    from nltk.corpus import stopwords as nltk_stopwords

    sw = set(nltk_stopwords.words("english"))
    out: list[str] = []
    for t in texts:
        t2 = normalize_text("" if t is None else str(t))
        out.append(tokenize_and_remove_stopwords(t2, sw))
    return out


@_dataclass_sm
class SentimentArtifacts:
    vectorizer: _TfidfVectorizer
    model: _LogisticRegression
    labels: list[str]


class SentimentModel:
    """Train and run emotion classification (13 labels) and derive sentiment groupings."""

    def __init__(self, model_dir: str = _os_sm.path.join("data", "models")) -> None:
        self.model_dir = model_dir
        self.artifacts: _Optional_sm[SentimentArtifacts] = None

    @property
    def model_path(self) -> str:
        return _os_sm.path.join(self.model_dir, "sentiment_model.pkl")

    def try_load(self) -> bool:
        if not _os_sm.path.exists(self.model_path):
            return False
        with open(self.model_path, "rb") as f:
            payload = _pickle_sm.load(f)
        self.artifacts = SentimentArtifacts(
            vectorizer=payload["vectorizer"],
            model=payload["model"],
            labels=payload["labels"],
        )
        return True

    def save(self) -> None:
        if self.artifacts is None:
            raise RuntimeError("No artifacts to save. Train first.")
        _os_sm.makedirs(self.model_dir, exist_ok=True)
        payload = {
            "vectorizer": self.artifacts.vectorizer,
            "model": self.artifacts.model,
            "labels": self.artifacts.labels,
        }
        with open(self.model_path, "wb") as f:
            _pickle_sm.dump(payload, f)

    def train(
        self,
        dataset_path: str = _os_sm.path.join("assets", "sentiment_data.csv"),
        sample_rows: _Optional_sm[int] = None,
        random_state: int = 42,
    ) -> None:
        df = _pd_sm.read_csv(dataset_path)
        required = {"ID", "text", "emotion"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Dataset missing required columns: {sorted(missing)}")

        if sample_rows is not None and sample_rows > 0 and len(df) > sample_rows:
            df = df.sample(n=sample_rows, random_state=random_state)

        df = df.copy()
        df["emotion_norm"] = df["emotion"].astype(str).str.strip().str.lower()
        df = df[df["emotion_norm"] != "neutral"]

        texts = preprocess_texts(df["text"].astype(str).tolist())
        y = df["emotion_norm"].astype(str).tolist()
        labels = sorted({v for v in y if v})
        if not labels:
            raise ValueError("No emotion labels found in dataset.")

        x_train, x_test, y_train, y_test = _train_test_split(
            texts, y, test_size=0.2, random_state=random_state, stratify=y
        )

        vectorizer = _TfidfVectorizer(max_features=10000)
        x_train_vec = vectorizer.fit_transform(x_train)
        x_test_vec = vectorizer.transform(x_test)

        model = _LogisticRegression(max_iter=800, solver="lbfgs", class_weight="balanced")
        model.fit(x_train_vec, y_train)

        preds = model.predict(x_test_vec)
        acc = _accuracy_score(y_test, preds)
        print(f"Accuracy: {acc:.4f}")
        print("Classification report:")
        print(_classification_report(y_test, preds, labels=labels, zero_division=0))
        print("Confusion matrix:")
        print(_confusion_matrix(y_test, preds, labels=labels))

        self.artifacts = SentimentArtifacts(vectorizer=vectorizer, model=model, labels=labels)

    def predict_emotion(self, text: str) -> str:
        if self.artifacts is None:
            return "neutral"
        cleaned = preprocess_texts([text])[0]
        x = self.artifacts.vectorizer.transform([cleaned])
        pred = self.artifacts.model.predict(x)[0]
        return str(pred).strip().lower() or "neutral"

    def predict(self, text: str) -> str:
        emotion = self.predict_emotion(text)
        return sentiment_from_emotion(emotion)

    def predict_group(self, text: str) -> str:
        emotion = self.predict_emotion(text)
        return emotion_group(emotion)


def emotion_group(emotion: str) -> str:
    e = (emotion or "").strip().lower()
    if e in {"love", "happiness", "fun", "enthusiasm", "relief"}:
        return "positive"
    if e in {"sadness", "hate", "anger", "worry"}:
        return "negative"
    return "low_engagement"


def sentiment_from_emotion(emotion: str) -> str:
    g = emotion_group(emotion)
    if g == "positive":
        return "positive"
    if g == "negative":
        return "negative"
    return "neutral"


In [ ]:
# Webcam emotion worker (inline from vision/webcam_emotion.py)

import time as _time_cv
from dataclasses import dataclass as _dataclass_cv
from typing import Optional as _Optional_cv

import cv2
import numpy as _np_cv


@_dataclass_cv(frozen=True)
class VisionConfig:
    camera_index: int = 0
    target_fps: float = 20.0
    preview_width: int = 300
    calibration_factor: float = 1.10
    smoothing_window: int = 30
    max_faces: int = 1


class WebcamEmotionWorker(StoppableThread):
    """Background thread for webcam + MediaPipe Face Mesh."""

    def __init__(
        self,
        shared: SharedState,
        face_image_path: str,
        camera_index: int = 0,
        target_fps: float = 20.0,
        smoothing_window: int = 30,
    ) -> None:
        super().__init__(daemon=True)
        self.shared = shared
        self.face_image_path = face_image_path
        self.cfg = VisionConfig(
            camera_index=camera_index,
            target_fps=target_fps,
            smoothing_window=smoothing_window,
        )
        self._calibration: _Optional_cv[Calibration] = None
        self._majority = RollingMajority(window_size=self.cfg.smoothing_window)
        self._mp_face_mesh = None
        self._mp_drawing = None
        self._mp_styles = None

        # Optional per-user calibration using a neutral face image (e.g. face.jpg).
        # Uncomment this helper and the call in run() if you want to compute
        # a baseline mouth width from a static image instead of using the
        # fixed threshold.
        #
        # def _compute_calibration_from_face(self) -> _Optional_cv[Calibration]:
        #     img_bgr = cv2.imread(self.face_image_path)
        #     if img_bgr is None or self._mp_face_mesh is None:
        #         return None
        #     img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        #     h, w = img_rgb.shape[:2]
        #     with self._mp_face_mesh.FaceMesh(
        #         static_image_mode=True,
        #         max_num_faces=1,
        #         refine_landmarks=True,
        #         min_detection_confidence=0.5,
        #     ) as fm:
        #         res = fm.process(img_rgb)
        #         if not res.multi_face_landmarks:
        #             return None
        #         face = res.multi_face_landmarks[0]
        #         xy = _np_cv.array([(lm.x, lm.y) for lm in face.landmark], dtype=_np_cv.float32)
        #         baseline = mouth_width_px(xy, (h, w))
        #         threshold = baseline * self.cfg.calibration_factor
        #         return Calibration(baseline_mouth_width=float(baseline), threshold=float(threshold))

    @staticmethod
    def _resize_for_preview(rgb: _np_cv.ndarray, preview_width: int) -> _np_cv.ndarray:
        h, w = rgb.shape[:2]
        if w == 0 or h == 0:
            return rgb
        scale = preview_width / float(w)
        new_h = max(1, int(h * scale))
        return cv2.resize(rgb, (preview_width, new_h), interpolation=cv2.INTER_AREA)

    @staticmethod
    def _probe_cameras(max_index: int = 4) -> list[int]:
        found: list[int] = []
        for idx in range(0, int(max_index) + 1):
            ok = False
            for backend in (cv2.CAP_DSHOW, cv2.CAP_MSMF, 0):
                try:
                    cap = cv2.VideoCapture(idx, backend)
                except Exception:
                    continue
                if cap.isOpened():
                    ok = True
                    try:
                        cap.release()
                    except Exception:
                        pass
                    break
                try:
                    cap.release()
                except Exception:
                    pass
            if ok:
                found.append(idx)
        return found or [0]

    def run(self) -> None:
        try:
            available = self._probe_cameras(max_index=4)
            self.shared.set_available_cameras(available)
        except Exception:
            self.shared.set_available_cameras([self.cfg.camera_index])

        try:
            from mediapipe.python.solutions import face_mesh as _mp_face_mesh
            from mediapipe.python.solutions import drawing_utils as _mp_drawing
            from mediapipe.python.solutions import drawing_styles as _mp_styles

            mp_face_mesh = _mp_face_mesh
            mp_drawing = _mp_drawing
            mp_styles = _mp_styles
        except Exception:
            try:
                import mediapipe as mp  # type: ignore

                mp_face_mesh = mp.solutions.face_mesh
                mp_drawing = mp.solutions.drawing_utils
                mp_styles = mp.solutions.drawing_styles
            except Exception:
                mp_face_mesh = None
                mp_drawing = None
                mp_styles = None

        self._mp_face_mesh = mp_face_mesh
        self._mp_drawing = mp_drawing
        self._mp_styles = mp_styles

        # Default fixed-threshold calibration.
        self._calibration = Calibration(baseline_mouth_width=0.0, threshold=65.0)
        # To enable calibration from a neutral reference photo (face.jpg),
        # uncomment the following lines and ensure the helper above is
        # uncommented as well:
        # calib = self._compute_calibration_from_face()
        # if calib is not None:
        #     self._calibration = calib

        cap: _Optional_cv[cv2.VideoCapture] = None
        active_index = None

        def open_camera(index: int) -> _Optional_cv[cv2.VideoCapture]:
            backends = [cv2.CAP_DSHOW, cv2.CAP_MSMF, 0]
            for backend in backends:
                try:
                    c = cv2.VideoCapture(int(index), backend)
                except Exception:
                    continue
                if c.isOpened():
                    return c
                try:
                    c.release()
                except Exception:
                    pass
            return None

        last_t = _time_cv.perf_counter()
        fps_window = []
        next_tick = _time_cv.perf_counter()
        tick_dt = 1.0 / max(1e-6, self.cfg.target_fps)

        if self._mp_face_mesh is not None:
            with self._mp_face_mesh.FaceMesh(
                static_image_mode=False,
                max_num_faces=self.cfg.max_faces,
                refine_landmarks=True,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5,
            ) as fm:
                while not self.stop_event.is_set():
                    now = _time_cv.perf_counter()
                    if now < next_tick:
                        _time_cv.sleep(min(0.005, next_tick - now))
                        continue
                    next_tick = now + tick_dt

                    desired = self.shared.get_requested_camera()
                    if cap is None or active_index != desired:
                        if cap is not None:
                            try:
                                cap.release()
                            except Exception:
                                pass
                            cap = None
                        cap = open_camera(desired)
                        active_index = desired if cap is not None else None

                    if cap is None:
                        snap = VisionSnapshot(
                            emotion_raw="Neutral",
                            emotion_smoothed="Neutral",
                            mouth_width=0.0,
                            landmarks_detected=0,
                            fps=0.0,
                            camera_index=int(desired),
                            camera_ok=False,
                            preview_rgb=None,
                        )
                        self.shared.update_vision(snap)
                        continue

                    ok, frame_bgr = cap.read()
                    if not ok or frame_bgr is None:
                        continue

                    frame_bgr = cv2.flip(frame_bgr, 1)
                    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                    h, w = frame_rgb.shape[:2]

                    res = fm.process(frame_rgb)
                    emotion_raw = "Neutral"
                    emotion_smoothed = "Neutral"
                    mouth_w = 0.0
                    landmarks_count = 0

                    overlay_rgb = frame_rgb.copy()

                    if res.multi_face_landmarks:
                        face = res.multi_face_landmarks[0]
                        landmarks_count = len(face.landmark)
                        xy = _np_cv.array([(lm.x, lm.y) for lm in face.landmark], dtype=_np_cv.float32)
                        mouth_w = mouth_width_px(xy, (h, w))
                        emotion_raw = classify_emotion(mouth_w, self._calibration.threshold)
                        self._majority.add(emotion_raw)
                        emotion_smoothed = self._majority.majority(default="Neutral")

                        self._mp_drawing.draw_landmarks(
                            image=overlay_rgb,
                            landmark_list=face,
                            connections=self._mp_face_mesh.FACEMESH_TESSELATION,
                            landmark_drawing_spec=None,
                            connection_drawing_spec=self._mp_styles.get_default_face_mesh_tesselation_style(),
                        )

                    dt = max(1e-6, now - last_t)
                    last_t = now
                    fps_window.append(1.0 / dt)
                    if len(fps_window) > 30:
                        fps_window.pop(0)
                    fps = float(sum(fps_window) / len(fps_window))

                    preview = self._resize_for_preview(overlay_rgb, self.cfg.preview_width)

                    snap = VisionSnapshot(
                        emotion_raw=emotion_raw,
                        emotion_smoothed=emotion_smoothed,
                        mouth_width=float(mouth_w),
                        landmarks_detected=int(landmarks_count),
                        fps=fps,
                        camera_index=int(active_index if active_index is not None else desired),
                        camera_ok=True,
                        preview_rgb=preview,
                    )
                    self.shared.update_vision(snap)
        else:
            while not self.stop_event.is_set():
                now = _time_cv.perf_counter()
                if now < next_tick:
                    _time_cv.sleep(min(0.005, next_tick - now))
                    continue
                next_tick = now + tick_dt

                desired = self.shared.get_requested_camera()
                if cap is None or active_index != desired:
                    if cap is not None:
                        try:
                            cap.release()
                        except Exception:
                            pass
                        cap = None
                    cap = open_camera(desired)
                    active_index = desired if cap is not None else None

                if cap is None:
                    snap = VisionSnapshot(
                        emotion_raw="Neutral",
                        emotion_smoothed="Neutral",
                        mouth_width=0.0,
                        landmarks_detected=0,
                        fps=0.0,
                        camera_index=int(desired),
                        camera_ok=False,
                        preview_rgb=None,
                    )
                    self.shared.update_vision(snap)
                    continue

                ok, frame_bgr = cap.read()
                if not ok or frame_bgr is None:
                    continue

                frame_bgr = cv2.flip(frame_bgr, 1)
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

                dt = max(1e-6, now - last_t)
                last_t = now
                fps_window.append(1.0 / dt)
                if len(fps_window) > 30:
                    fps_window.pop(0)
                fps = float(sum(fps_window) / len(fps_window))

                preview = self._resize_for_preview(frame_rgb, self.cfg.preview_width)

                snap = VisionSnapshot(
                    emotion_raw="Neutral",
                    emotion_smoothed="Neutral",
                    mouth_width=0.0,
                    landmarks_detected=0,
                    fps=fps,
                    camera_index=int(active_index if active_index is not None else desired),
                    camera_ok=True,
                    preview_rgb=preview,
                )
                self.shared.update_vision(snap)

        if cap is not None:
            cap.release()


In [ ]:
# Pygame UI (inline from game/pygame_ui.py)

import math as _math_pg
import os as _os_pg
import random as _random_pg
import time as _time_pg
from dataclasses import dataclass as _dataclass_pg
from typing import Optional as _Optional_pg

import numpy as _np_pg
import pygame as _pygame


WINDOW_W, WINDOW_H = 1280, 720
HUD_H = 80
SIDE_W = 300
BOARD_SIZE = 500


@_dataclass_pg
class CellAnim:
    kind: str
    started_s: float
    duration_s: float = 0.30

    def t(self, now_s: float) -> float:
        if self.duration_s <= 0:
            return 1.0
        return max(0.0, min(1.0, (now_s - self.started_s) / self.duration_s))


class PygameApp:
    def __init__(self, shared: SharedState, sentiment: SentimentModel) -> None:
        self.shared = shared
        self.sentiment = sentiment

        _pygame.init()
        _pygame.display.set_caption("Emotion-Aware Tic-Tac-Toe")
        self.screen = _pygame.display.set_mode((WINDOW_W, WINDOW_H))
        self.clock = _pygame.time.Clock()

        self.font_title = _pygame.font.SysFont("segoe ui", 28, bold=True)
        self.font_ui = _pygame.font.SysFont("segoe ui", 18)
        self.font_small = _pygame.font.SysFont("segoe ui", 14)

        self.rng = _random_pg.Random(7)
        self.particles = ParticleSystem()

        self.board = Board()
        self.turn = "X"
        self.winner: _Optional_pg[str] = None
        self.draw = False
        self.win_line: _Optional_pg[list[tuple[int, int]]] = None

        self.cell_anims: dict[tuple[int, int], CellAnim] = {}
        self.cell_pulse: dict[tuple[int, int], Tween] = {}
        self.board_shake = Tween(duration_s=0.45, easing=ease_in_out)
        self.board_shake.done = True

        self.neutral_player_turns = 0
        self.hint_active = False
        self.hint_cell: _Optional_pg[tuple[int, int]] = None
        self.hint_slide = Tween(duration_s=0.35, easing=ease_out_back)
        self.hint_slide.done = True

        self.theme = NEUTRAL_THEME
        self.theme_target = NEUTRAL_THEME
        self.theme_tween = Tween(duration_s=0.40, easing=ease_in_out)
        self.theme_tween.done = True

        self.ai_thinking_until_s = 0.0
        self.ai_pending_move: _Optional_pg[tuple[int, int]] = None
        self.ai_typing = ""
        self.ai_typing_until_s = 0.0

        self.sfx_move = try_load_sound(_os_pg.path.join("assets", "sounds", "move.wav"))
        self.sfx_win = try_load_sound(_os_pg.path.join("assets", "sounds", "win.wav"))
        self.sfx_click = try_load_sound(_os_pg.path.join("assets", "sounds", "click.wav"))

        self.scene = "play"
        self.feedback_text = ""
        self.last_sentiment = "neutral"
        self.postgame_message = ""

    def run(self) -> None:
        while True:
            dt_s = self.clock.tick(60) / 1000.0
            if not self._handle_events():
                return

            self._step(dt_s)
            self._draw()
            _pygame.display.flip()

    def _layout(self) -> dict[str, _pygame.Rect]:
        hud = _pygame.Rect(0, 0, WINDOW_W, HUD_H)
        left = _pygame.Rect(0, HUD_H, SIDE_W, WINDOW_H - HUD_H)
        right = _pygame.Rect(WINDOW_W - SIDE_W, HUD_H, SIDE_W, WINDOW_H - HUD_H)
        center = _pygame.Rect(SIDE_W, HUD_H, WINDOW_W - 2 * SIDE_W, WINDOW_H - HUD_H)

        board_rect = _pygame.Rect(0, 0, BOARD_SIZE, BOARD_SIZE)
        board_rect.center = center.center
        return {"hud": hud, "left": left, "right": right, "center": center, "board": board_rect}

    def _handle_events(self) -> bool:
        for ev in _pygame.event.get():
            if ev.type == _pygame.QUIT:
                return False
            if ev.type == _pygame.KEYDOWN and ev.key == _pygame.K_ESCAPE:
                return False
            if ev.type == _pygame.KEYDOWN and ev.key == _pygame.K_c:
                idx = self.shared.request_next_camera()
                self.shared.set_dialogue(f"Switching camera to index {idx} (press C to cycle).", ttl_s=2.5)

            if self.scene == "play":
                if ev.type == _pygame.MOUSEBUTTONDOWN and ev.button == 1:
                    self._on_click(_pygame.mouse.get_pos())
            else:
                self._handle_postgame_input(ev)
        return True

    def _handle_postgame_input(self, ev: _pygame.event.Event) -> None:
        if ev.type != _pygame.KEYDOWN:
            return
        if ev.key == _pygame.K_RETURN:
            emotion = self.sentiment.predict_emotion(self.feedback_text)
            self.shared.set_feedback_emotion(emotion)
            sentiment = sentiment_from_emotion(emotion)
            self.last_sentiment = sentiment

            _, _, _, tuning, _ = self.shared.get_snapshot()
            tuning.adjust_for_sentiment(sentiment)

            behavior = self.shared.last_behavior
            self.shared.set_dialogue(behavior.system_message, ttl_s=4.0)
            self._emit_ui_effect(behavior.ui_effect)
            self.feedback_text = ""
            self._start_new_match()
            return
        if ev.key == _pygame.K_BACKSPACE:
            self.feedback_text = self.feedback_text[:-1]
            return
        if ev.unicode and ev.unicode.isprintable():
            if len(self.feedback_text) < 140:
                self.feedback_text += ev.unicode

    def _on_click(self, pos: tuple[int, int]) -> None:
        if self.board.game_over() or self.turn != "X":
            return
        rects = self._layout()
        board_rect = rects["board"]
        if not board_rect.collidepoint(pos):
            return
        cell = self._cell_from_pos(pos, board_rect)
        if cell is None:
            return
        r, c = cell
        if not self.board.place(r, c, "X"):
            return

        now_s = _time_pg.perf_counter()
        self.cell_anims[(r, c)] = CellAnim(kind="X", started_s=now_s)
        self.cell_pulse[(r, c)] = Tween(duration_s=0.20, easing=ease_out_back)
        if self.sfx_move:
            self.sfx_move.play()

        emotion, _, stats, _, _ = self.shared.get_snapshot()
        stats.record_emotion(emotion)

        if (r, c) == (1, 1):
            self.particles.burst(board_rect.center, self._confetti_palette(emotion), count=18)

        self._post_move_updates(player="X")
        self.turn = "O"
        self._queue_ai_move()

    def _queue_ai_move(self) -> None:
        emotion, _, _, tuning, _ = self.shared.get_snapshot()
        decision = decide_move(self.board, emotion, tuning, self.rng)
        self.shared.ai_mode = decision.mode
        self.shared.set_dialogue(decision.message, ttl_s=3.0)

        self.ai_pending_move = decision.move
        self.ai_thinking_until_s = _time_pg.perf_counter() + decision.thinking_delay_s
        self.ai_typing = ""
        self.ai_typing_until_s = _time_pg.perf_counter() + 1.2

    def _post_move_updates(self, player: str) -> None:
        self.winner = self.board.winner()
        self.draw = self.board.is_draw()
        self.win_line = Board.winning_line_coords(self.board.grid)
        if self.winner or self.draw:
            self._finish_match()

    def _finish_match(self) -> None:
        emotion, _, stats, _, _ = self.shared.get_snapshot()
        stats.games_played += 1
        if self.winner == "X":
            stats.wins += 1
            if emotion == "Happy":
                self.postgame_message = "Great victory! You're improving fast."
            else:
                self.postgame_message = "Nice win. Keep building momentum."
        elif self.winner == "O":
            stats.losses += 1
            if emotion == "Neutral":
                self.postgame_message = "That was a tough round. Want a tip for next time?"
            else:
                self.postgame_message = "Close one. Ready for another?"
        else:
            stats.draws += 1
            self.postgame_message = "Draw! Well played."

        if self.winner and self.sfx_win:
            self.sfx_win.play()

        rects = self._layout()
        if self.winner:
            self.particles.burst(rects["board"].center, self._confetti_palette(emotion), count=60)
        else:
            self.board_shake.reset()

        self.scene = "postgame"
        self.turn = "X"
        self.ai_pending_move = None
        self.ai_thinking_until_s = 0.0

    def _start_new_match(self) -> None:
        self.board = Board()
        self.turn = "X"
        self.winner = None
        self.draw = False
        self.win_line = None
        self.cell_anims.clear()
        self.cell_pulse.clear()
        self.neutral_player_turns = 0
        self.hint_active = False
        self.hint_cell = None
        self.scene = "play"
        self._set_theme_target(self.shared.current_emotion)

    def _step(self, dt_s: float) -> None:
        emotion, vision, _, _, ai_mode = self.shared.get_snapshot()
        self._set_theme_target(emotion)
        self._step_theme(dt_s)
        self.particles.step(dt_s)

        if self.scene == "play":
            if self.turn == "O" and not self.board.game_over():
                now = _time_pg.perf_counter()
                if now >= self.ai_thinking_until_s and self.ai_pending_move is not None:
                    r, c = self.ai_pending_move
                    if self.board.place(r, c, "O"):
                        self.cell_anims[(r, c)] = CellAnim(kind="O", started_s=now)
                        if self.sfx_move:
                            self.sfx_move.play()
                        self.shared.set_dialogue(self._random_ai_dialogue(emotion), ttl_s=2.5)
                        self._post_move_updates(player="O")
                    self.ai_pending_move = None
                    self.turn = "X"

            self._update_hints(emotion)
            self._typing_effect()

    def _typing_effect(self) -> None:
        if self.turn != "O" or self.scene != "play" or self.board.game_over():
            return
        now = _time_pg.perf_counter()
        if now > self.ai_typing_until_s:
            return
        target = "Analyzing best move..."
        if len(self.ai_typing) < len(target):
            self.ai_typing += target[len(self.ai_typing)]

    def _update_hints(self, emotion: str) -> None:
        if self.scene != "play" or self.turn != "X" or self.board.game_over():
            return
        if emotion == "Neutral":
            self.neutral_player_turns = min(99, self.neutral_player_turns + 1)
        else:
            self.neutral_player_turns = 0
            self.hint_active = False
            self.hint_cell = None

        if self.neutral_player_turns >= 3 and not self.hint_active:
            self.hint_active = True
            self.hint_cell = self._suggest_hint_cell()
            self.hint_slide.reset()
            self.shared.set_dialogue("Hint: The center square is often the strongest opening move.", ttl_s=4.0)

    def _suggest_hint_cell(self) -> _Optional_pg[tuple[int, int]]:
        if self.board.grid[1][1] == "":
            return (1, 1)
        for cell in [(0, 0), (0, 2), (2, 0), (2, 2), (0, 1), (1, 0), (1, 2), (2, 1)]:
            r, c = cell
            if self.board.grid[r][c] == "":
                return cell
        return None

    def _set_theme_target(self, emotion: str) -> None:
        target = HAPPY_THEME if emotion == "Happy" else NEUTRAL_THEME
        try:
            ui_theme = self.shared.last_behavior.ui_theme
        except Exception:
            ui_theme = ""
        if ui_theme:
            target = EMOTION_THEMES.get(ui_theme, target)
        if target.name != self.theme_target.name:
            self.theme_target = target
            self.theme_tween.reset()

    def _emit_ui_effect(self, effect: str) -> None:
        rects = self._layout()
        center = rects["board"].center
        eff = (effect or "").strip().lower()
        if eff == "hearts":
            palette = [_pygame.Color(255, 110, 180), _pygame.Color(255, 170, 215), _pygame.Color(255, 70, 140)]
            self.particles.burst(center, palette, count=42)
        elif eff in {"particles", "dynamic_board"}:
            palette = [self.theme.accent, self.theme.accent_soft, _pygame.Color(255, 235, 150)]
            self.particles.burst(center, palette, count=60)
        elif eff == "flash":
            palette = [_pygame.Color(255, 255, 255), _pygame.Color(200, 220, 255)]
            self.particles.burst(center, palette, count=24)
        elif eff in {"calm_bg", "calming_overlay", "slow_fade", "subtle_pulse", "supportive_theme", "dynamic_lighting", "board_glow"}:
            palette = [self.theme.accent_soft, self.theme.accent]
            self.particles.burst(center, palette, count=18)

    def _step_theme(self, dt_s: float) -> None:
        if self.theme_tween.done:
            self.theme = self.theme_target
            return
        t = self.theme_tween.step(dt_s)
        self.theme = Theme(
            name=self.theme_target.name,
            accent=lerp_color(self.theme.accent, self.theme_target.accent, t),
            accent_soft=lerp_color(self.theme.accent_soft, self.theme_target.accent_soft, t),
            bg_top=lerp_color(self.theme.bg_top, self.theme_target.bg_top, t),
            bg_bottom=lerp_color(self.theme.bg_bottom, self.theme_target.bg_bottom, t),
            board_glow=lerp_color(self.theme.board_glow, self.theme_target.board_glow, t),
        )

    def _draw(self) -> None:
        rects = self._layout()
        draw_vertical_gradient(self.screen, self.screen.get_rect(), self.theme.bg_top, self.theme.bg_bottom)

        self._draw_hud(rects["hud"])
        self._draw_webcam_panel(rects["left"])
        self._draw_status_panel(rects["right"])
        self._draw_board(rects["board"])
        self.particles.draw(self.screen)

        if self.scene == "postgame":
            self._draw_postgame_overlay()

    def _draw_hud(self, rect: _pygame.Rect) -> None:
        _pygame.draw.rect(self.screen, _pygame.Color(0, 0, 0, 80), rect)
        emotion, _, stats, _, ai_mode = self.shared.get_snapshot()

        title = self.font_title.render("Emotion-Aware Tic-Tac-Toe", True, _pygame.Color(240, 245, 255))
        self.screen.blit(title, (16, 18))

        feedback_emotion = (self.shared.last_feedback_emotion or "none").lower()
        hud_text = (
            f"Face: {emotion}    "
            f"Feedback: {feedback_emotion} ({self.last_sentiment})    "
            f"AI Mode: {ai_mode}    "
            f"Games Played: {stats.games_played}    Wins: {stats.wins}"
        )
        t = self.font_ui.render(hud_text, True, _pygame.Color(200, 210, 230))
        self.screen.blit(t, (420, 28))

        msg = self.shared.get_dialogue()
        if msg:
            m = self.font_small.render(msg, True, self.theme.accent_soft)
            self.screen.blit(m, (16, 52))

    def _draw_webcam_panel(self, rect: _pygame.Rect) -> None:
        emotion, vision, _, _, _ = self.shared.get_snapshot()
        border = self.theme.accent if emotion == "Happy" else self.theme.accent_soft
        _pygame.draw.rect(self.screen, _pygame.Color(18, 22, 30), rect, border_radius=12)
        _pygame.draw.rect(self.screen, border, rect, width=2, border_radius=12)

        pad = 12
        inner = rect.inflate(-2 * pad, -2 * pad)

        header = self.font_ui.render("WEBCAM FEED", True, _pygame.Color(235, 240, 255))
        self.screen.blit(header, (inner.left, inner.top))

        preview_height = int(inner.height * 0.55)
        preview_rect = _pygame.Rect(inner.left, inner.top + 26, inner.width, max(40, preview_height))
        if isinstance(vision.preview_rgb, _np_pg.ndarray):
            frame = vision.preview_rgb
            surf = self._rgb_to_surface(frame)
            surf = _pygame.transform.smoothscale(surf, (preview_rect.width, preview_rect.height))
            self.screen.blit(surf, preview_rect.topleft)
        else:
            txt = self.font_small.render("Camera unavailable (press C to switch)", True, _pygame.Color(170, 180, 200))
            self.screen.blit(txt, (preview_rect.left, preview_rect.top + 10))

        info_y = preview_rect.bottom + 6
        cam_line = self.font_small.render(
            f"Camera: {vision.camera_index}  ({'OK' if vision.camera_ok else 'NOT FOUND'})",
            True,
            _pygame.Color(170, 180, 200),
        )
        info1 = self.font_small.render(f"Landmarks: {vision.landmarks_detected}", True, _pygame.Color(170, 180, 200))
        info2 = self.font_small.render(f"Mouth width: {vision.mouth_width:.2f}", True, _pygame.Color(170, 180, 200))
        info3 = self.font_small.render(f"Emotion: {vision.emotion_smoothed}  ({vision.fps:.1f} FPS)", True, border)
        self.screen.blit(cam_line, (inner.left, info_y))
        self.screen.blit(info1, (inner.left, info_y + 18))
        self.screen.blit(info2, (inner.left, info_y + 36))
        self.screen.blit(info3, (inner.left, info_y + 54))

    def _draw_status_panel(self, rect: _pygame.Rect) -> None:
        emotion, _, stats, tuning, ai_mode = self.shared.get_snapshot()
        _pygame.draw.rect(self.screen, _pygame.Color(18, 22, 30), rect, border_radius=12)
        _pygame.draw.rect(self.screen, self.theme.accent_soft, rect, width=2, border_radius=12)

        pad = 12
        x, y = rect.left + pad, rect.top + pad
        title = self.font_ui.render("AI STATUS", True, _pygame.Color(235, 240, 255))
        self.screen.blit(title, (x, y))
        y += 30

        lines = [
            f"AI Status: {'Thinking...' if self.turn == 'O' and self.scene=='play' and not self.board.game_over() else 'Idle'}",
            f"Face emotion: {emotion}",
            f"Feedback emotion: {(self.shared.last_feedback_emotion or 'none').lower()}",
            f"Last sentiment: {self.last_sentiment}",
            f"Difficulty: {ai_mode} Mode",
            f"Assistive mistake prob: {min(0.30, max(0.20, tuning.assistive_mistake_prob)):.2f}",
            f"Cameras: {self.shared.camera_status_summary()}",
        ]
        for line in lines:
            t = self.font_small.render(line, True, _pygame.Color(170, 180, 200))
            self.screen.blit(t, (x, y))
            y += 18

        y += 10
        if self.ai_typing:
            typing = self.font_small.render(self.ai_typing, True, self.theme.accent_soft)
            self.screen.blit(typing, (x, y))
            y += 24

        if self.hint_active and self.hint_cell is not None:
            slide = self.hint_slide.step(1.0 / 60.0) if not self.hint_slide.done else 1.0
            hint_x = int(lerp(rect.right + 180, x, slide))
            hint = self.font_small.render("Tip: Controlling the center helps.", True, self.theme.accent_soft)
            self.screen.blit(hint, (hint_x, rect.bottom - 48))

        y = rect.bottom - 72
        dist = stats.emotion_counts
        total = max(1, dist.get("Happy", 0) + dist.get("Neutral", 0))
        h_pct = int(100 * dist.get("Happy", 0) / total)
        n_pct = 100 - h_pct
        d1 = self.font_small.render(f"Emotion dist: Happy {h_pct}% / Neutral {n_pct}%", True, _pygame.Color(170, 180, 200))
        self.screen.blit(d1, (x, y))

    def _draw_board(self, rect: _pygame.Rect) -> None:
        emotion, _, _, _, _ = self.shared.get_snapshot()

        shake_off = _pygame.Vector2(0, 0)
        if not self.board_shake.done:
            t = self.board_shake.step(1.0 / 60.0)
            amp = (1.0 - t) * 10.0
            shake_off.x = _math_pg.sin(_time_pg.perf_counter() * 55.0) * amp
            shake_off.y = _math_pg.cos(_time_pg.perf_counter() * 42.0) * amp

        rect2 = rect.move(int(shake_off.x), int(shake_off.y))

        glow = _pygame.Surface((rect2.width + 16, rect2.height + 16), _pygame.SRCALPHA)
        _pygame.draw.rect(glow, _pygame.Color(self.theme.board_glow.r, self.theme.board_glow.g, self.theme.board_glow.b, 60), glow.get_rect(), border_radius=24)
        self.screen.blit(glow, (rect2.left - 8, rect2.top - 8))

        _pygame.draw.rect(self.screen, _pygame.Color(22, 26, 36), rect2, border_radius=18)
        _pygame.draw.rect(self.screen, self.theme.accent_soft, rect2, width=2, border_radius=18)

        for i in range(1, 3):
            x = rect2.left + i * rect2.width // 3
            _pygame.draw.line(self.screen, _pygame.Color(60, 70, 90), (x, rect2.top + 18), (x, rect2.bottom - 18), 4)
            y = rect2.top + i * rect2.height // 3
            _pygame.draw.line(self.screen, _pygame.Color(60, 70, 90), (rect2.left + 18, y), (rect2.right - 18, y), 4)

        if self.scene == "play" and self.turn == "X" and not self.board.game_over():
            mx, my = _pygame.mouse.get_pos()
            if rect2.collidepoint((mx, my)):
                cell = self._cell_from_pos((mx, my), rect2)
                if cell and self.board.grid[cell[0]][cell[1]] == "":
                    cell_rect = self._cell_rect(cell, rect2)
                    col = self.theme.accent if emotion == "Happy" else self.theme.accent_soft
                    self._draw_cell_glow(cell_rect, col)

        if self.hint_active and self.hint_cell is not None:
            cell_rect = self._cell_rect(self.hint_cell, rect2)
            self._draw_cell_glow(cell_rect, self.theme.accent_soft)

        now_s = _time_pg.perf_counter()
        for r in range(3):
            for c in range(3):
                mark = self.board.grid[r][c]
                if not mark:
                    continue
                cell_rect = self._cell_rect((r, c), rect2)
                anim = self.cell_anims.get((r, c))
                t = anim.t(now_s) if anim else 1.0
                if mark == "X":
                    self._draw_x(cell_rect, t, emotion)
                else:
                    self._draw_o(cell_rect, t, emotion)

                pulse = self.cell_pulse.get((r, c))
                if pulse and not pulse.done:
                    p = pulse.step(1.0 / 60.0)
                    s = 1.0 + 0.10 * p
                    pr = cell_rect.copy()
                    pr.width = int(pr.width * s)
                    pr.height = int(pr.height * s)
                    pr.center = cell_rect.center
                    _pygame.draw.rect(self.screen, _pygame.Color(255, 255, 255, 0), pr, border_radius=12)

        if self.win_line:
            pts = [self._cell_rect(cell, rect2).center for cell in self.win_line]
            _pygame.draw.line(self.screen, self.theme.accent, pts[0], pts[-1], 10)

    def _draw_postgame_overlay(self) -> None:
        overlay = _pygame.Surface((WINDOW_W, WINDOW_H), _pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 160))
        self.screen.blit(overlay, (0, 0))

        box = _pygame.Rect(0, 0, 760, 320)
        box.center = (WINDOW_W // 2, WINDOW_H // 2)
        _pygame.draw.rect(self.screen, _pygame.Color(20, 24, 34), box, border_radius=18)
        _pygame.draw.rect(self.screen, self.theme.accent_soft, box, width=2, border_radius=18)

        emotion, _, stats, _, _ = self.shared.get_snapshot()
        header = "Game Over"
        if self.winner == "X":
            header = "You Win!"
        elif self.winner == "O":
            header = "AI Wins"
        elif self.draw:
            header = "Draw"

        t1 = self.font_title.render(header, True, _pygame.Color(240, 245, 255))
        self.screen.blit(t1, (box.left + 24, box.top + 18))

        msg = self.font_ui.render(self.postgame_message, True, self.theme.accent_soft)
        self.screen.blit(msg, (box.left + 24, box.top + 60))

        q = self.font_ui.render("How did you feel about this game?", True, _pygame.Color(200, 210, 230))
        self.screen.blit(q, (box.left + 24, box.top + 110))

        input_rect = _pygame.Rect(box.left + 24, box.top + 148, box.width - 48, 44)
        _pygame.draw.rect(self.screen, _pygame.Color(12, 14, 20), input_rect, border_radius=10)
        _pygame.draw.rect(self.screen, self.theme.accent, input_rect, width=2, border_radius=10)

        typed = self.feedback_text or ""
        t_in = self.font_ui.render(typed + ("|" if int(_time_pg.time() * 2) % 2 == 0 else ""), True, _pygame.Color(230, 235, 245))
        self.screen.blit(t_in, (input_rect.left + 12, input_rect.top + 10))

        hint = self.font_small.render("Press Enter to submit and start next match.", True, _pygame.Color(170, 180, 200))
        self.screen.blit(hint, (box.left + 24, box.bottom - 34))

        dist = stats.emotion_counts
        total = max(1, dist.get("Happy", 0) + dist.get("Neutral", 0))
        h_pct = int(100 * dist.get("Happy", 0) / total)
        n_pct = 100 - h_pct
        stats_line = f"Games Played: {stats.games_played}   Wins: {stats.wins}   Losses: {stats.losses}   Draws: {stats.draws}"
        stats2 = f"Emotion Distribution: Happy {h_pct}% / Neutral {n_pct}%"
        s1 = self.font_small.render(stats_line, True, _pygame.Color(170, 180, 200))
        s2 = self.font_small.render(stats2, True, _pygame.Color(170, 180, 200))
        self.screen.blit(s1, (box.left + 24, box.top + 210))
        self.screen.blit(s2, (box.left + 24, box.top + 232))

    def _cell_from_pos(self, pos: tuple[int, int], board_rect: _pygame.Rect) -> _Optional_pg[tuple[int, int]]:
        x, y = pos
        if not board_rect.collidepoint(pos):
            return None
        rel_x = x - board_rect.left
        rel_y = y - board_rect.top
        c = int(rel_x / (board_rect.width / 3))
        r = int(rel_y / (board_rect.height / 3))
        if 0 <= r < 3 and 0 <= c < 3:
            return (r, c)
        return None

    def _cell_rect(self, cell: tuple[int, int], board_rect: _pygame.Rect) -> _pygame.Rect:
        r, c = cell
        cw = board_rect.width // 3
        ch = board_rect.height // 3
        x = board_rect.left + c * cw
        y = board_rect.top + r * ch
        return _pygame.Rect(x + 8, y + 8, cw - 16, ch - 16)

    def _draw_cell_glow(self, rect: _pygame.Rect, color: _pygame.Color) -> None:
        glow = _pygame.Surface((rect.width, rect.height), _pygame.SRCALPHA)
        _pygame.draw.rect(glow, _pygame.Color(color.r, color.g, color.b, 70), glow.get_rect(), border_radius=14)
        self.screen.blit(glow, rect.topleft)
        _pygame.draw.rect(self.screen, _pygame.Color(color.r, color.g, color.b, 180), rect, width=2, border_radius=14)

    def _draw_x(self, rect: _pygame.Rect, t: float, emotion: str) -> None:
        col = self.theme.accent if emotion == "Happy" else self.theme.accent_soft
        pad = 18
        a = (rect.left + pad, rect.top + pad)
        b = (rect.right - pad, rect.bottom - pad)
        c = (rect.left + pad, rect.bottom - pad)
        d = (rect.right - pad, rect.top + pad)

        if t < 0.5:
            tt = ease_in_cubic(t / 0.5)
            p = (int(a[0] + (b[0] - a[0]) * tt), int(a[1] + (b[1] - a[1]) * tt))
            _pygame.draw.line(self.screen, col, a, p, 10)
        else:
            _pygame.draw.line(self.screen, col, a, b, 10)
            tt = ease_in_cubic((t - 0.5) / 0.5)
            p = (int(c[0] + (d[0] - c[0]) * tt), int(c[1] + (d[1] - c[1]) * tt))
            _pygame.draw.line(self.screen, col, c, p, 10)

    def _draw_o(self, rect: _pygame.Rect, t: float, emotion: str) -> None:
        col = self.theme.accent if emotion == "Happy" else self.theme.accent_soft
        center = rect.center
        radius = min(rect.width, rect.height) // 2 - 14
        start_angle = -_np_pg.pi / 2
        end_angle = start_angle + (_np_pg.pi * 2) * ease_in_out(t)
        _pygame.draw.arc(
            self.screen,
            col,
            _pygame.Rect(center[0] - radius, center[1] - radius, radius * 2, radius * 2),
            start_angle,
            end_angle,
            10,
        )

    def _confetti_palette(self, emotion: str) -> list[_pygame.Color]:
        if emotion == "Happy":
            return [_pygame.Color(70, 255, 150), _pygame.Color(240, 230, 90), _pygame.Color(80, 200, 120)]
        return [_pygame.Color(120, 190, 255), _pygame.Color(160, 220, 255), _pygame.Color(80, 160, 255)]

    def _random_ai_dialogue(self, emotion: str) -> str:
        if emotion == "Happy":
            return self.rng.choice(["Nice move!", "You’re playing well!", "Keep it up!"])
        return self.rng.choice(["You're doing great. Keep going.", "Remember to block your opponent.", "Take your time."])

    @staticmethod
    def _rgb_to_surface(rgb: _np_pg.ndarray) -> _pygame.Surface:
        arr = _np_pg.transpose(rgb, (1, 0, 2))
        return _pygame.surfarray.make_surface(arr)


In [ ]:
# Orchestration: run_app (inline from runtime.py)

import os as _os_rt
import time as _time_rt


def run_app() -> None:
    shared = SharedState()

    model_dir = _os_rt.path.join("data", "models")
    _os_rt.makedirs(model_dir, exist_ok=True)
    sentiment = SentimentModel(model_dir=model_dir)
    loaded = sentiment.try_load()
    if not loaded:
        try:
            dataset_path = _os_rt.path.join("assets", "sentiment_data.csv")
            sentiment.train(dataset_path=dataset_path, sample_rows=20000)
            sentiment.save()
        except Exception:
            pass

    vision_worker = WebcamEmotionWorker(
        shared=shared,
        face_image_path=_os_rt.path.join("data", "face.jpg"),
        camera_index=0,
        target_fps=20.0,
        smoothing_window=30,
    )
    vision_worker.start()

    try:
        app = PygameApp(shared=shared, sentiment=sentiment)
        app.run()
    finally:
        vision_worker.stop()
        vision_worker.join(timeout=2.0)
        _time_rt.sleep(0.05)


In [ ]:
# Run the full Emotion-Aware Tic-Tac-Toe app

# NOTE: This opens a Pygame window. In Colab this only works with a
# local runtime or tools that can forward a display.

run_app()